# FinSight – Machine Learning Credit Risk Scoring & Scorecard Modeling
**Project:** Financial Analytics and Risk Reporting System (Nexvia Internship Evaluation)
**Methodology:** Lending Club Portfolio Modeling, Weight of Evidence (WOE), Information Value (IV), Logistic Regression, and FICO Scorecard Calibration.

In [1]:
import os, sys
sys.path.append('..')
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ml.woe import WOETransformer
from ml.scorecard import CreditScorecard
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report

conn = sqlite3.connect('../finsight.db')
df = pd.read_sql('SELECT * FROM loans l JOIN borrowers b ON l.borrower_id=b.borrower_id', conn)
print(f'Loaded {len(df)} loans. Overall Default Rate: {(df["is_default"].mean()*100):.2f}%')

## 1. Information Value (IV) & Feature Selection
Evaluate predictive strength of borrower credit attributes via Weight of Evidence (WOE).

In [2]:
features = ['loan_grade', 'interest_rate', 'debt_to_income_ratio', 'revolving_utilization_pct', 'derogatory_records', 'annual_income']
X = df[features]
y = df['is_default']

woe = WOETransformer(n_bins=5)
woe.fit(X, y)
display(woe.get_woe_summary())

## 2. Logistic Regression Model Training & True Evaluation
Train Logistic Regression classifier on WOE-transformed features.

In [3]:
X_woe = woe.transform(X)
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_woe, y)
y_probs = clf.predict_proba(X_woe)[:, 1]
auc = roc_auc_score(y, y_probs)
print(f'Model Full Portfolio ROC-AUC: {auc:.4f}')

## 3. Scorecard Scaling (Basel II / FICO Formula)
Scale probabilities into standardized 300-850 scores: Score = Offset + Factor * ln(Odds).

In [4]:
card = CreditScorecard(target_score=650, target_odds=20.0, pdo=20)
scores = card.calculate_score(y_probs)
df['credit_score'] = scores
df['risk_band'] = [card.get_risk_band(s) for s in scores]
print('Score distribution by Risk Band:')
display(df.groupby('risk_band')['credit_score'].agg(['count', 'min', 'mean', 'max']))